# RTT-Graph Connection Pipeline

Implements the pipeline from the *RTT-Graph Connection Guide* (Parvez, 2026-09-23).

Two datasets stay separate until Phase 5/6:
- **Topology graphs** (`graph_01.csv` .. `graph_15.csv`): edges = direct physical links.
- **RTT measurements** (`caida_real_rtt_supervised.csv`): endpoint pairs = end-to-end pings, usually many hops apart.

Hard rules enforced throughout:
- RTT is never written onto a single edge, and never split evenly across a path's edges.
- The graph-ID <-> ITDK-ID mapping is never invented; if it can't be recovered, the pipeline stops and reports the gap (Phase 5).
- Unmapped RTT rows are saved, never silently dropped.
- Shortest paths are computed only inside the specific one of the 15 sampled graphs that contains both endpoints, never in the full ITDK topology.
- Inferred paths are tagged `path_method = "shortest_path_inferred"`, never labeled as measured traceroutes.


In [10]:
from pathlib import Path
import re
import pandas as pd
import networkx as nx

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Colab - skipping drive.mount(). Make sure the paths below are reachable locally.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# --- Drive locations (DeepNT / DeepNet+ thesis project) ---

PROJECT_ROOT = Path("/content/drive/My Drive/Thesis/DeepNT_Output")
GRAPH_DIR = PROJECT_ROOT / "graphs"
MODEL_OUTPUT_DIR = GRAPH_DIR / "model_output"
RTT_DIR = PROJECT_ROOT / "real_rtt"
SUPERVISED_RTT_FILE = RTT_DIR / "caida_real_rtt_supervised.csv"

DATASET_DIR = Path("/content/drive/My Drive/DataSet")
NODES_FILE = DATASET_DIR / "midar-iff-snmp.nodes"
LINKS_FILE = DATASET_DIR / "midar-iff-snmp.links"
IP_NODE_FILE = DATASET_DIR / "itdk_rtt_ip_to_node.csv"

print("Path sanity check:")
for label, p in [
    ("PROJECT_ROOT", PROJECT_ROOT),
    ("GRAPH_DIR", GRAPH_DIR),
    ("RTT_DIR", RTT_DIR),
    ("SUPERVISED_RTT_FILE", SUPERVISED_RTT_FILE),
    ("DATASET_DIR", DATASET_DIR),
    ("NODES_FILE", NODES_FILE),
    ("LINKS_FILE", LINKS_FILE),
    ("IP_NODE_FILE", IP_NODE_FILE),
]:
    mark = "OK     " if p.exists() else "MISSING"
    print(f"  [{mark}] {label:22s} {p}")


Path sanity check:
  [OK     ] PROJECT_ROOT           /content/drive/My Drive/Thesis/DeepNT_Output
  [OK     ] GRAPH_DIR              /content/drive/My Drive/Thesis/DeepNT_Output/graphs
  [OK     ] RTT_DIR                /content/drive/My Drive/Thesis/DeepNT_Output/real_rtt
  [OK     ] SUPERVISED_RTT_FILE    /content/drive/My Drive/Thesis/DeepNT_Output/real_rtt/caida_real_rtt_supervised.csv
  [OK     ] DATASET_DIR            /content/drive/My Drive/DataSet
  [OK     ] NODES_FILE             /content/drive/My Drive/DataSet/midar-iff-snmp.nodes
  [OK     ] LINKS_FILE             /content/drive/My Drive/DataSet/midar-iff-snmp.links
  [OK     ] IP_NODE_FILE           /content/drive/My Drive/DataSet/itdk_rtt_ip_to_node.csv


## Phase 1 — Load graphs

Read all `graph_XX.csv` into NetworkX graphs. Always safe to run.


In [12]:
NODE_ID_RE = re.compile(r'^N\d+$', re.IGNORECASE)

def load_graph_csv(path: Path) -> nx.Graph:
    """Load an edge-list CSV (e.g. 'N1,N240' per row) into an undirected graph,
    auto-detecting whether the file has a header row."""
    df = pd.read_csv(path, header=None, dtype=str)
    if df.shape[1] < 2:
        raise ValueError(f"{path.name}: expected at least 2 columns, got {df.shape[1]}")

    first_row = df.iloc[0, :2]
    looks_like_header = not all(NODE_ID_RE.match(str(v).strip()) for v in first_row)
    if looks_like_header:
        df = df.iloc[1:].reset_index(drop=True)

    src_col, dst_col = df.columns[0], df.columns[1]
    G = nx.Graph()
    G.add_edges_from(zip(
        df[src_col].astype(str).str.strip(),
        df[dst_col].astype(str).str.strip(),
    ))
    return G


graph_files = sorted(GRAPH_DIR.glob("graph_*.csv"))
if not graph_files:
    raise FileNotFoundError(f"No graph_*.csv files found under {GRAPH_DIR}")

graphs = {}
for f in graph_files:
    graph_id = f.stem  # e.g. 'graph_01'
    graphs[graph_id] = load_graph_csv(f)

graph_node_ids = set()
for G in graphs.values():
    graph_node_ids.update(G.nodes())

print(f"Loaded {len(graphs)} graphs")
for gid, G in graphs.items():
    print(f"  {gid}: {G.number_of_nodes():>6,} nodes, {G.number_of_edges():>6,} edges")
print(f"Union of graph node IDs: {len(graph_node_ids):,}")


Loaded 15 graphs
  graph_01:    500 nodes,    596 edges
  graph_02:    500 nodes,    632 edges
  graph_03:    500 nodes,    581 edges
  graph_04:    500 nodes,    617 edges
  graph_05:    500 nodes,    596 edges
  graph_06:    500 nodes,    587 edges
  graph_07:    500 nodes,    627 edges
  graph_08:    500 nodes,    595 edges
  graph_09:    500 nodes,    601 edges
  graph_10:    500 nodes,    627 edges
  graph_11:    500 nodes,    577 edges
  graph_12:    500 nodes,    625 edges
  graph_13:    500 nodes,    544 edges
  graph_14:    500 nodes,    570 edges
  graph_15:    500 nodes,    599 edges
Union of graph node IDs: 500


## Phase 2 — Load RTT

Read the RTT CSV, auto-detect the endpoint and RTT columns, and confirm the RTT unit
(milliseconds vs microseconds) before trusting any value downstream.


In [4]:
def detect_column(columns, keywords, prefer_tokens=(), exclude=()):
    """Find a column whose name, split into '_'/space tokens, has a token matching
    a keyword. Matching is done per-token (never across a word boundary), so e.g.
    'start_time' can never match keyword 'rtt' just because stripping the underscore
    would spell 'starttime'. `prefer_tokens` lets a caller break ties between several
    candidate columns (e.g. prefer 'destination_rtt_ms' over 'final_responding_hop_rtt_ms')."""
    cols = [c for c in columns if c not in exclude]

    def tokens(col):
        return re.split(r'[_\s]+', str(col).lower())

    if prefer_tokens:
        for col in cols:
            toks = tokens(col)
            has_kw = any(kw in tok for tok in toks for kw in keywords)
            has_pref = any(pt in tok for tok in toks for pt in prefer_tokens)
            if has_kw and has_pref:
                return col

    for col in cols:
        toks = tokens(col)
        if any(kw in tok for tok in toks for kw in keywords):
            return col
    return None


rtt_raw = pd.read_csv(SUPERVISED_RTT_FILE)
print(f"Loaded {SUPERVISED_RTT_FILE.name}: {len(rtt_raw):,} rows, columns = {list(rtt_raw.columns)}")

src_col = detect_column(rtt_raw.columns, ['src', 'source', 'from'])
dst_col = detect_column(rtt_raw.columns, ['dst', 'dest', 'target', 'to'])
rtt_col = detect_column(rtt_raw.columns, ['rtt', 'delay', 'latency'], prefer_tokens=['destination', 'dst', 'end'])

if src_col is None or dst_col is None or rtt_col is None:
    raise ValueError(
        f"Could not auto-detect endpoint/RTT columns from {list(rtt_raw.columns)}. "
        f"Detected so far: src={src_col!r}, dst={dst_col!r}, rtt={rtt_col!r}. "
        "Adjust the keyword lists above."
    )
print(f"Detected columns -> src: '{src_col}', dst: '{dst_col}', rtt: '{rtt_col}'")

rtt_df = rtt_raw[[src_col, dst_col, rtt_col]].copy()
rtt_df.columns = ['src_node', 'dst_node', 'rtt_raw']
rtt_df['src_node'] = rtt_df['src_node'].astype(str).str.strip()
rtt_df['dst_node'] = rtt_df['dst_node'].astype(str).str.strip()

rtt_df['rtt_raw'] = pd.to_numeric(rtt_df['rtt_raw'], errors='coerce')
n_before_numeric = len(rtt_df)
rtt_df = rtt_df.dropna(subset=['rtt_raw'])
n_after_numeric = len(rtt_df)
if n_after_numeric == 0:
    raise ValueError(f"Column '{rtt_col}' produced no numeric values at all - wrong column detected?")
if n_after_numeric < 0.5 * n_before_numeric:
    print(f"WARNING: {n_before_numeric - n_after_numeric:,} of {n_before_numeric:,} rows dropped as "
          f"non-numeric RTT - double check rtt_col='{rtt_col}' is really an RTT column.")

median_rtt = rtt_df['rtt_raw'].median()
if median_rtt > 1000:
    detected_unit = 'microseconds'
    rtt_df['rtt_ms'] = rtt_df['rtt_raw'] / 1000.0
else:
    detected_unit = 'milliseconds'
    rtt_df['rtt_ms'] = rtt_df['rtt_raw']

print(f"Median raw RTT value: {median_rtt:.3f} -> inferred unit: {detected_unit}")
print(">>> MANUAL GATE: eyeball a handful of raw values below and confirm this inference before proceeding.")
print(rtt_df[['src_node', 'dst_node', 'rtt_raw', 'rtt_ms']].head(10))


Loaded caida_real_rtt_supervised.csv: 51,072 rows, columns = ['measurement_id', 'warts_file', 'source_ip', 'destination_ip', 'start_time', 'hop_count_reported', 'responding_hop_count', 'hop_numbers', 'path_ips', 'destination_responded', 'destination_hop_number', 'destination_rtt_ms', 'final_responding_hop_ip', 'final_responding_hop_number', 'final_responding_hop_rtt_ms']
Detected columns -> src: 'source_ip', dst: 'destination_ip', rtt: 'destination_rtt_ms'
Median raw RTT value: 218208.500 -> inferred unit: microseconds
>>> MANUAL GATE: eyeball a handful of raw values below and confirm this inference before proceeding.
       src_node        dst_node   rtt_raw   rtt_ms
0  137.50.19.17     87.66.59.45   33493.0   33.493
1  137.50.19.17     88.47.76.65   41176.0   41.176
2  137.50.19.17   184.85.59.103   14548.0   14.548
3  137.50.19.17   180.80.102.23  277705.0  277.705
4  137.50.19.17  43.245.109.190  194505.0  194.505
5  137.50.19.17   46.214.180.26   50476.0   50.476
6  137.50.19.17  

### Phase 2 (continued) — translate IP endpoints to ITDK node IDs

This RTT file's endpoints are raw IPs (`source_ip` / `destination_ip`), not node IDs —
CAIDA traceroute measurements are recorded by IP. They must be resolved to ITDK node
IDs via `itdk_rtt_ip_to_node.csv` before anything downstream reasons about nodes;
skipping this would make Phase 4 silently compare graph node IDs against raw IP
strings and report 0% overlap everywhere. Rows whose IP has no known ITDK node are
saved, never dropped silently. If a future RTT file already uses node IDs, this cell
is a no-op.


In [5]:
IPV4_RE = re.compile(r'^\d{1,3}(\.\d{1,3}){3}$')

def looks_like_ip(series, sample=200):
    sample_vals = series.dropna().astype(str).head(sample)
    if len(sample_vals) == 0:
        return False
    return (sample_vals.str.match(IPV4_RE)).mean() > 0.9


src_is_ip = looks_like_ip(rtt_df['src_node'])
dst_is_ip = looks_like_ip(rtt_df['dst_node'])
print(f"Endpoint format check -> src looks like raw IP: {src_is_ip}, dst looks like raw IP: {dst_is_ip}")

if src_is_ip or dst_is_ip:
    if not IP_NODE_FILE.exists():
        raise FileNotFoundError(
            f"RTT endpoints are raw IPs but the IP->node mapping file is missing: {IP_NODE_FILE}"
        )

    ip_map_raw = pd.read_csv(IP_NODE_FILE)
    print(f"Loaded {IP_NODE_FILE.name}: columns = {list(ip_map_raw.columns)}, rows = {len(ip_map_raw):,}")

    ip_col = detect_column(ip_map_raw.columns, ['ip'])
    node_col = detect_column(ip_map_raw.columns, ['node', 'itdk'], exclude=(ip_col,))
    if ip_col is None or node_col is None:
        raise ValueError(
            f"Could not detect IP/node columns in {IP_NODE_FILE.name} "
            f"(ip_col={ip_col!r}, node_col={node_col!r}, columns={list(ip_map_raw.columns)})"
        )
    print(f"IP->node mapping columns -> ip: '{ip_col}', node: '{node_col}'")

    ip_to_node = dict(zip(
        ip_map_raw[ip_col].astype(str).str.strip(),
        ip_map_raw[node_col].astype(str).str.strip(),
    ))

    rtt_df['src_ip'] = rtt_df['src_node']
    rtt_df['dst_ip'] = rtt_df['dst_node']
    if src_is_ip:
        rtt_df['src_node'] = rtt_df['src_ip'].map(ip_to_node)
    if dst_is_ip:
        rtt_df['dst_node'] = rtt_df['dst_ip'].map(ip_to_node)

    unmapped_ip_mask = rtt_df['src_node'].isna() | rtt_df['dst_node'].isna()
    n_unmapped_ip = int(unmapped_ip_mask.sum())
    n_total_ip = len(rtt_df)

    MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    unmapped_ip_out = MODEL_OUTPUT_DIR / "unmapped_rtt_ip_rows.csv"
    rtt_df[unmapped_ip_mask].to_csv(unmapped_ip_out, index=False)

    rtt_df = rtt_df[~unmapped_ip_mask].copy()
    print(f"IP->node translation: {n_unmapped_ip:,} of {n_total_ip:,} rows had an unmapped endpoint IP "
          f"(saved to {unmapped_ip_out}); {len(rtt_df):,} rows usable.")
else:
    print("Endpoints do not look like raw IPs - treating src_node/dst_node as already being node IDs.")

rtt_node_ids = set(rtt_df['src_node']) | set(rtt_df['dst_node'])
print(f"\nRTT rows: {len(rtt_df):,}, unique endpoint node IDs: {len(rtt_node_ids):,}")


Endpoint format check -> src looks like raw IP: True, dst looks like raw IP: True
Loaded itdk_rtt_ip_to_node.csv: columns = ['ip_address', 'node_id'], rows = 11,364
IP->node mapping columns -> ip: 'ip_address', node: 'node_id'
IP->node translation: 41,057 of 51,072 rows had an unmapped endpoint IP (saved to /content/drive/My Drive/Thesis/DeepNT_Output/graphs/model_output/unmapped_rtt_ip_rows.csv); 10,015 rows usable.

RTT rows: 10,015, unique endpoint node IDs: 10,018


## Phase 3 — Load ITDK

Stream the raw `.nodes` file line by line for real ITDK node IDs. The file is never
read into memory as a whole (no `.readlines()`, no `pd.read_csv` on it) — **and the
full ITDK ID universe is never materialized either**, only the file reads are
streamed. The full ITDK topology has tens of millions of nodes; building a Python
`set()` of all of them is exactly what crashed the Colab kernel with an out-of-memory
kill (no traceback — that's the signature of a runtime OOM, not a Python exception).

Instead, this phase checks each streamed ITDK node ID against the small set of IDs we
actually care about — the union of `graph_node_ids` (Phase 1) and `rtt_node_ids`
(Phase 2), a few thousand entries — and keeps only the matches. Memory use stays
bounded by that small set, not by the size of ITDK.


In [6]:
def stream_itdk_node_ids(path: Path):
    """Yield ITDK node IDs (e.g. 'N136561') one line at a time from a .nodes file
    without ever materializing the whole file in memory."""
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            if not line or line[0] == '#':
                continue
            if not line.startswith('node'):
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            token = parts[1]
            if token.endswith(':'):
                token = token[:-1]
            if token.startswith('N'):
                yield token


target_ids = graph_node_ids | rtt_node_ids
print(f"Streaming {NODES_FILE.name}, checking each ID against {len(target_ids):,} known "
      f"graph/RTT IDs (the full ITDK ID universe is never held in memory).")

itdk_matches_graph = set()
itdk_matches_rtt = set()
itdk_sample_ids = []
itdk_total_count = 0

for node_id in stream_itdk_node_ids(NODES_FILE):
    itdk_total_count += 1
    if len(itdk_sample_ids) < 5:
        itdk_sample_ids.append(node_id)
    if node_id in target_ids:
        if node_id in graph_node_ids:
            itdk_matches_graph.add(node_id)
        if node_id in rtt_node_ids:
            itdk_matches_rtt.add(node_id)
    if itdk_total_count % 5_000_000 == 0:
        print(f"  ...processed {itdk_total_count:,} node lines so far "
              f"({len(itdk_matches_graph):,} graph / {len(itdk_matches_rtt):,} RTT matches so far)")

print(f"ITDK file: {itdk_total_count:,} node lines total (full ID set not retained).")
print(f"  graph IDs found in ITDK: {len(itdk_matches_graph):,} / {len(graph_node_ids):,}")
print(f"  RTT IDs found in ITDK:   {len(itdk_matches_rtt):,} / {len(rtt_node_ids):,}")
print(f"  sample ITDK IDs seen:    {itdk_sample_ids}")


Streaming midar-iff-snmp.nodes, checking each ID against 10,511 known graph/RTT IDs (the full ITDK ID universe is never held in memory).
  ...processed 5,000,000 node lines so far (500 graph / 90 RTT matches so far)
  ...processed 10,000,000 node lines so far (500 graph / 113 RTT matches so far)
  ...processed 15,000,000 node lines so far (500 graph / 125 RTT matches so far)
  ...processed 20,000,000 node lines so far (500 graph / 129 RTT matches so far)
  ...processed 25,000,000 node lines so far (500 graph / 129 RTT matches so far)
  ...processed 30,000,000 node lines so far (500 graph / 129 RTT matches so far)
  ...processed 35,000,000 node lines so far (500 graph / 130 RTT matches so far)
  ...processed 40,000,000 node lines so far (500 graph / 130 RTT matches so far)
  ...processed 45,000,000 node lines so far (500 graph / 133 RTT matches so far)
  ...processed 50,000,000 node lines so far (500 graph / 135 RTT matches so far)
  ...processed 55,000,000 node lines so far (500 graph 

## Phase 4 — Diagnose

Count overlaps between all three ID sets. This must run, and be read, before any
merge code is written — it decides which case (A/B/C/D) Phase 5 is dealing with.

The graph<->ITDK and RTT<->ITDK overlaps were already computed during Phase 3's
streaming pass (`itdk_matches_graph` / `itdk_matches_rtt`) precisely so this phase
never needs the full ITDK ID set either.


In [7]:
def print_overlap(name_a, total_a, name_b, total_b, inter, sample_n=5):
    n = len(inter)
    pct_a = 100 * n / total_a if total_a else 0.0
    pct_b = 100 * n / total_b if total_b else 0.0
    print(f"{name_a} ({total_a:,}) vs {name_b} ({total_b:,}):")
    print(f"  overlap = {n:,}  ({pct_a:.2f}% of {name_a}, {pct_b:.2f}% of {name_b})")
    if n:
        print(f"  sample overlap IDs: {list(inter)[:sample_n]}")
    print()


print("=" * 70)
print("PHASE 4: ID NAMESPACE DIAGNOSTIC")
print("=" * 70)
graph_rtt_overlap = graph_node_ids & rtt_node_ids
print_overlap("graph IDs", len(graph_node_ids), "RTT IDs", len(rtt_node_ids), graph_rtt_overlap)
print_overlap("graph IDs", len(graph_node_ids), "ITDK IDs", itdk_total_count, itdk_matches_graph)
print_overlap("RTT IDs", len(rtt_node_ids), "ITDK IDs", itdk_total_count, itdk_matches_rtt)

if graph_node_ids and len(graph_rtt_overlap) == len(graph_node_ids):
    id_case = "A"
    print(">>> Case A: graph IDs are a subset of / identical to RTT IDs -> direct merge possible.")
elif itdk_matches_graph or itdk_matches_rtt:
    id_case = "B_or_C"
    print(">>> Case B/C: no direct graph<->RTT overlap, but ITDK overlap exists on at least one side")
    print("    -> a graph-ID <-> ITDK-ID mapping is needed (Phase 5).")
else:
    id_case = "D"
    print(">>> Case D: no usable overlap found in any pairing -> Phase 5 must STOP, do not merge.")

print(f"\nid_case = {id_case!r}")


PHASE 4: ID NAMESPACE DIAGNOSTIC
graph IDs (500) vs RTT IDs (10,018):
  overlap = 7  (1.40% of graph IDs, 0.07% of RTT IDs)
  sample overlap IDs: ['N16', 'N66', 'N219', 'N1', 'N134']

graph IDs (500) vs ITDK IDs (169,358,424):
  overlap = 500  (100.00% of graph IDs, 0.00% of ITDK IDs)
  sample overlap IDs: ['N76', 'N296', 'N70', 'N276', 'N286']

RTT IDs (10,018) vs ITDK IDs (169,358,424):
  overlap = 10,018  (100.00% of RTT IDs, 0.01% of ITDK IDs)
  sample overlap IDs: ['N166419594', 'N166823025', 'N147100631', 'N116227572', 'N157626289']

>>> Case B/C: no direct graph<->RTT overlap, but ITDK overlap exists on at least one side
    -> a graph-ID <-> ITDK-ID mapping is needed (Phase 5).

id_case = 'B_or_C'


## Phase 5 — Resolve mapping

Find or recover the graph-ID <-> ITDK-ID mapping. This is the only phase allowed to
fail and stop the pipeline — if it can't confirm a mapping, that is correct behavior,
not a bug to route around. Nothing here invents or guesses a mapping.


In [8]:
print("=" * 70)
print("PHASE 5: RESOLVE GRAPH-ID <-> ITDK-ID MAPPING")
print("=" * 70)

id_map = None

if id_case == "A":
    id_map = {nid: nid for nid in graph_node_ids}
    print(f"Case A confirmed by Phase 4: using identity mapping ({len(id_map):,} entries).")
else:
    if IP_NODE_FILE.exists():
        ip_map_df = pd.read_csv(IP_NODE_FILE)
        print(f"Loaded {IP_NODE_FILE.name}: columns = {list(ip_map_df.columns)}, rows = {len(ip_map_df):,}")

        candidate_graph_cols = [c for c in ip_map_df.columns if 'graph' in c.lower()]
        candidate_itdk_cols = [c for c in ip_map_df.columns
                                if ('node' in c.lower() or 'itdk' in c.lower()) and c not in candidate_graph_cols]

        if candidate_graph_cols and candidate_itdk_cols:
            gcol, icol = candidate_graph_cols[0], candidate_itdk_cols[0]
            id_map = dict(zip(
                ip_map_df[gcol].astype(str).str.strip(),
                ip_map_df[icol].astype(str).str.strip(),
            ))
            print(f"Built id_map from '{gcol}' -> '{icol}' ({len(id_map):,} entries).")
        else:
            print(f"{IP_NODE_FILE.name} does not expose a graph-ID column directly")
            print("(it maps IPs to ITDK node IDs, not graph-local IDs to ITDK IDs).")
    else:
        print(f"MISSING: {IP_NODE_FILE} not found.")

    if id_map is None:
        print("\nSearching GRAPH_DIR for a separate node-mapping file...")
        mapping_candidates = sorted(set(
            list(GRAPH_DIR.glob("*map*.csv")) + list(GRAPH_DIR.glob("*node*id*.csv"))
        ))
        if mapping_candidates:
            print(f"Found candidate mapping file(s): {[p.name for p in mapping_candidates]}")
            print("MANUAL STEP: inspect these and adapt Phase 5 to build id_map from them before proceeding.")
        else:
            print("No mapping file found in GRAPH_DIR.")

if id_map is None:
    print("\n" + "!" * 70)
    print("STOP: no graph-ID <-> ITDK-ID mapping could be confirmed.")
    print("This is expected/correct behavior per the pipeline contract - do NOT invent one.")
    print("Recover it from the original graph-generation code (the script that produced")
    print("graph_01.csv .. graph_15.csv should have preserved, for every relabeled node,")
    print("which original ITDK node it came from), or supply a mapping file and rerun this cell.")
    print("!" * 70)
else:
    print(f"\nid_map resolved with {len(id_map):,} entries. Proceeding to Phase 6.")


PHASE 5: RESOLVE GRAPH-ID <-> ITDK-ID MAPPING
Loaded itdk_rtt_ip_to_node.csv: columns = ['ip_address', 'node_id'], rows = 11,364
itdk_rtt_ip_to_node.csv does not expose a graph-ID column directly
(it maps IPs to ITDK node IDs, not graph-local IDs to ITDK IDs).

Searching GRAPH_DIR for a separate node-mapping file...
No mapping file found in GRAPH_DIR.

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
STOP: no graph-ID <-> ITDK-ID mapping could be confirmed.
This is expected/correct behavior per the pipeline contract - do NOT invent one.
Recover it from the original graph-generation code (the script that produced
graph_01.csv .. graph_15.csv should have preserved, for every relabeled node,
which original ITDK node it came from), or supply a mapping file and rerun this cell.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


## Phase 6 — Translate + merge

Apply `id_map` to the RTT endpoints, find the shortest path *within the specific
sampled graph* that contains both endpoints, and attach the RTT to that whole path.
Unmapped/unreachable rows are saved to their own file, never dropped silently.

Guarded on Phase 5: this cell only merges if `id_map` was resolved.


In [ ]:
import numpy as np

RTT_AGG = "median"  # robust to traceroute RTT outliers; use "mean" for a plain average
PATH_METHOD = "shortest_path_inferred"
OUT_COLS = ['graph_id', 'source', 'destination', 'rtt_ms', 'measurement_count',
            'path', 'path_edges', 'path_length', 'path_method']


def merge_rtt_into_graphs(rtt_df, graphs, id_map_by_graph, agg=RTT_AGG):
    """Attach RTT to the inferred shortest path inside each graph.

    id_map_by_graph: {graph_id: {graph_node: itdk_node}}
    Returns (merged_df, unmapped_df). RTT is attached to whole paths only, and every
    pair that yields no path row is kept in unmapped_df with a reason.
    """
    # one row per directed ITDK pair; measurement_count = raw observations aggregated
    pairs = (rtt_df.groupby(['src_node', 'dst_node'], as_index=False)
                   .agg(rtt_ms=('rtt_ms', agg), measurement_count=('rtt_ms', 'size')))
    pairs['mappable'] = False
    placed = set()
    merged, unmapped = [], []

    for graph_id, G in graphs.items():
        graph_to_itdk = {g: i for g, i in id_map_by_graph.get(graph_id, {}).items() if g in G}
        to_graph = {i: g for g, i in graph_to_itdk.items()}
        if len(to_graph) != len(graph_to_itdk):
            raise ValueError(f"{graph_id}: id_map is not one-to-one (two graph nodes share an ITDK ID)")

        p = pairs.assign(source=pairs['src_node'].map(to_graph),
                         destination=pairs['dst_node'].map(to_graph))
        ok = p['source'].notna() & p['destination'].notna() & (p['source'] != p['destination'])
        pairs['mappable'] |= ok

        for src, grp in p[ok].groupby('source'):
            paths = nx.single_source_shortest_path(G, src)  # one BFS per source, not per pair
            for r in grp.itertuples():
                path = paths.get(r.destination)
                if path is None:
                    unmapped.append({'graph_id': graph_id, 'src_node': r.src_node, 'dst_node': r.dst_node,
                                     'rtt_ms': r.rtt_ms, 'measurement_count': r.measurement_count,
                                     'reason': 'no_path_in_graph'})
                    continue
                placed.add(r.Index)
                merged.append({
                    'graph_id': graph_id, 'source': src, 'destination': r.destination,
                    'rtt_ms': r.rtt_ms, 'measurement_count': r.measurement_count,
                    'path': '|'.join(path),
                    'path_edges': '|'.join(f"{a}-{b}" for a, b in zip(path, path[1:])),
                    'path_length': len(path) - 1, 'path_method': PATH_METHOD,
                })

    lost = pairs.loc[~pairs['mappable'], ['src_node', 'dst_node', 'rtt_ms', 'measurement_count']].copy()
    lost['graph_id'] = None
    lost['reason'] = np.where(lost['src_node'] == lost['dst_node'],
                              'src_equals_dst', 'endpoint_not_in_graph_mapping')

    merged_df = pd.DataFrame(merged, columns=OUT_COLS)
    unmapped_df = pd.concat([pd.DataFrame(unmapped), lost], ignore_index=True)
    return merged_df, unmapped_df


print("=" * 70)
print("PHASE 6: TRANSLATE + MERGE")
print("=" * 70)

if id_map is None:
    print("Skipped: Phase 5 did not resolve a mapping. Fix Phase 5 first, then rerun this cell.")
    merged_df = unmapped_df = pd.DataFrame()
else:
    # nested {graph_id: {...}} = per-graph mapping; flat {graph_node: itdk_node} = shared by all graphs
    by_graph = id_map if all(isinstance(v, dict) for v in id_map.values()) else {g: id_map for g in graphs}
    merged_df, unmapped_df = merge_rtt_into_graphs(rtt_df, graphs, by_graph)

    MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    merged_out = MODEL_OUTPUT_DIR / "path_level_rtt_dataset.csv"
    unmapped_out = MODEL_OUTPUT_DIR / "unmapped_rtt_rows.csv"
    merged_df.to_csv(merged_out, index=False)
    unmapped_df.to_csv(unmapped_out, index=False)

    n_pairs = len(rtt_df[['src_node', 'dst_node']].drop_duplicates())
    print(f"RTT: {len(rtt_df):,} raw rows -> {n_pairs:,} unique directed pairs")
    print(f"Merged path-level rows: {len(merged_df):,} -> {merged_out}")
    print(f"Unmapped entries:       {len(unmapped_df):,} -> {unmapped_out}")
    if not unmapped_df.empty:
        print(unmapped_df['reason'].value_counts().to_string())

merged_df.head(10)
